<h2 align='center'>Finding best model and hyper parameter tunning using GridSearchCV</h2>

**For iris flower dataset in sklearn library, we are going to find out best model and best hyper parameters using GridSearchCV**

**Load iris flower dataset**

In [1]:
import numpy as np
from sklearn import svm, datasets
iris = datasets.load_iris()

In [2]:
import pandas as pd
df = pd.DataFrame(iris.data,columns=iris.feature_names)
df['flower'] = iris.target
df['flower'] = df['flower'].apply(lambda x: iris.target_names[x])
df[47:150]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),flower
47,4.6,3.2,1.4,0.2,setosa
48,5.3,3.7,1.5,0.2,setosa
49,5.0,3.3,1.4,0.2,setosa
50,7.0,3.2,4.7,1.4,versicolor
51,6.4,3.2,4.5,1.5,versicolor
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


<h3 style='color:blue'>Approach 1: Use train_test_split and manually tune parameters by trial and error</h3>

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.3)

In [4]:
model = svm.SVC(kernel='rbf',C=30,gamma='auto')
model.fit(X_train,y_train)
model.score(X_test, y_test)

0.9333333333333333

<h3 style='color:blue'>Approach 2: Use K Fold Cross validation</h3>

**Manually try suppling models with different parameters to cross_val_score function with 5 fold cross validation**

In [5]:
from sklearn.model_selection import cross_val_score

In [6]:
cross_val_score(svm.SVC(kernel='linear',C=10,gamma='auto'), X_train, y_train, cv=5)

array([1.        , 1.        , 0.9047619 , 0.95238095, 0.95238095])

In [7]:
cross_val_score(svm.SVC(kernel='rbf',C=10,gamma='auto'), X_train, y_train, cv=5)

array([1.        , 1.        , 0.9047619 , 0.95238095, 0.95238095])

In [8]:
cross_val_score(svm.SVC(kernel='rbf',C=20,gamma='auto'), X_train, y_train, cv=5)

array([1.        , 1.        , 0.9047619 , 0.95238095, 0.95238095])

**Above approach is tiresome and very manual. We can use for loop as an alternative**

In [9]:
kernels = ['rbf', 'linear']
C = [1,10,20]
avg_scores = {}
for kval in kernels:
    for cval in C:
        cv_scores = cross_val_score(svm.SVC(kernel=kval,C=cval,gamma='auto'), X_train, y_train, cv=5)
        avg_scores[kval + '_' + str(cval)] = np.average(cv_scores)

avg_scores

{'rbf_1': 0.9714285714285715,
 'rbf_10': 0.9619047619047618,
 'rbf_20': 0.9619047619047618,
 'linear_1': 0.9714285714285715,
 'linear_10': 0.9619047619047618,
 'linear_20': 0.9523809523809523}

**From above results we can say that rbf with C=1 or linear with C=1 will give best performance**

<h3 style='color:blue'>Approach 3: Use GridSearchCV</h3>

**GridSearchCV does exactly same thing as for loop above but in a single line of code**

In [10]:
from sklearn.model_selection import GridSearchCV
clf = GridSearchCV(svm.SVC(gamma='auto'), {
    'C': [1,10,20],
    'kernel': ['rbf','linear']
}, cv=5, return_train_score=False)

In [11]:
clf.fit(X_train, y_train)
clf.cv_results_

{'mean_fit_time': array([0.003128, 0.      , 0.      , 0.      , 0.      , 0.      ]),
 'std_fit_time': array([0.00625601, 0.        , 0.        , 0.        , 0.        ,
        0.        ]),
 'mean_score_time': array([0.        , 0.        , 0.00300913, 0.        , 0.00313945,
        0.        ]),
 'std_score_time': array([0.        , 0.        , 0.00601826, 0.        , 0.0062789 ,
        0.        ]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'C': 1, 'kernel': 'rbf'},
  {'C': 1, 'kernel': 'linear'},
  {'C': 10, 'kernel': 'rbf'},
  {'C': 10, 'kernel': 'linear'},
  {'C': 20, 'kernel': 'rbf'},
  {'C': 20, 'kernel': 'linear'}],
 'split0_test_score'

In [12]:
df = pd.DataFrame(clf.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003128,0.006256,0.000000,0.000000,1,rbf,"{'C': 1, 'kernel': 'rbf'}",1.0,0.952381,1.000000,0.952381,0.952381,0.971429,0.023328,1
1,0.000000,0.000000,0.000000,0.000000,1,linear,"{'C': 1, 'kernel': 'linear'}",1.0,0.952381,1.000000,0.952381,0.952381,0.971429,0.023328,1
2,0.000000,0.000000,0.003009,0.006018,10,rbf,"{'C': 10, 'kernel': 'rbf'}",1.0,1.000000,0.904762,0.952381,0.952381,0.961905,0.035635,3
3,0.000000,0.000000,0.000000,0.000000,10,linear,"{'C': 10, 'kernel': 'linear'}",1.0,1.000000,0.904762,0.952381,0.952381,0.961905,0.035635,3
4,0.000000,0.000000,0.003139,0.006279,20,rbf,"{'C': 20, 'kernel': 'rbf'}",1.0,1.000000,0.904762,0.952381,0.952381,0.961905,0.035635,3
5,0.000000,0.000000,0.000000,0.000000,20,linear,"{'C': 20, 'kernel': 'linear'}",1.0,0.952381,0.904762,0.952381,0.952381,0.952381,0.030117,6


In [13]:
df[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.971429
1,1,linear,0.971429
2,10,rbf,0.961905
3,10,linear,0.961905
4,20,rbf,0.961905
5,20,linear,0.952381


In [14]:
dir(clf)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_check_feature_names',
 '_check_n_features',
 '_check_refit_for_multimetric',
 '_estimator_type',
 '_format_results',
 '_get_default_requests',
 '_get_metadata_request',
 '_get_param_names',
 '_get_tags',
 '_more_tags',
 '_parameter_constraints',
 '_repr_html_',
 '_repr_html_inner',
 '_repr_mimebundle_',
 '_required_parameters',
 '_run_search',
 '_select_best_index',
 '_validate_data',
 '_validate_params',
 'best_estimator_',
 'best_index_',
 'best_params_',
 'best_score_',
 '

In [15]:
clf.best_params_

{'C': 1, 'kernel': 'rbf'}

In [16]:
clf.best_score_

0.9714285714285715

**Use RandomizedSearchCV to reduce number of iterations and with random combination of parameters. This is useful when you have too many parameters to try and your training time is longer. It helps reduce the cost of computation**

In [17]:
from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(svm.SVC(gamma='auto'), {
        'C': [1,10,20],
        'kernel': ['rbf','linear']
    }, 
    cv=5, 
    return_train_score=False, 
    n_iter=2
)

In [18]:
rs.fit(X_train, y_train)
pd.DataFrame(rs.cv_results_)[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,20,linear,0.952381
1,10,rbf,0.961905


**How about different models with different hyperparameters?**

In [19]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

model_params = {
    'svm': {
        'model': svm.SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }  
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params' : {
            'n_estimators': [1,5,10]
        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(solver='liblinear',multi_class='auto'),
        'params': {
            'C': [1,5,10]
        }
    }
}


In [20]:
scores = []

for model_name, mp in model_params.items():
    clf =  GridSearchCV(mp['model'], mp['params'], cv=5, return_train_score=False)
    clf.fit(X_train, y_train)
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })
    
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df

,model,best_score,best_params
0,svm,0.971429,"{'C': 1, 'kernel': 'rbf'}"
1,random_forest,0.942857,{'n_estimators': 5}
2,logistic_regression,0.952381,{'C': 10}


**Based on above, I can conclude that SVM with C=1 and kernel='rbf' is the best model for solving my problem of iris flower classification**